# Phase 3 — Ton Transformer

On construit le modèle **pièce par pièce**, en testant chaque pièce. À la fin de ce notebook, tu auras un vrai GPT d'environ 46 millions de paramètres… qui ne sait encore rien (il n'est pas entraîné : c'est la phase 4).

**Type d'exécution : CPU suffit.** On ne fait que construire et tester, pas entraîner.

Rappel de ce qu'on a vu : une table de comptes ne peut pas regarder loin, car le nombre de contextes explose. Le Transformer règle ça avec deux idées :
1. **les embeddings** : chaque token devient un vecteur, et des tokens qui se ressemblent ont des vecteurs proches. Ce qu'il apprend sur « chat » sert pour « chien » ;
2. **l'attention** : pour prédire le token suivant, chaque position peut aller « regarder » toutes les positions précédentes et y piocher l'information utile.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
DOSSIER = "/content/drive/MyDrive/petit-llm"

!pip -q install tokenizers
import math, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(f"{DOSSIER}/tokenizer.json")
torch.manual_seed(0)
print("Tokenizer chargé :", tokenizer.get_vocab_size(), "tokens")

Mounted at /content/drive
Tokenizer chargé : 16384 tokens


## 1. Les embeddings : des tokens aux vecteurs

Un embedding, c'est simplement **une grande table** : une ligne de 512 nombres par token. Le token 4521 → la ligne 4521.

Au départ, ces nombres sont aléatoires. Pendant l'entraînement, ils vont bouger pour que les tokens utilisés dans des contextes semblables aient des vecteurs proches.

On ajoute un **deuxième embedding pour la position** (1re place, 2e place…) : sans lui, le modèle verrait les mots comme un sac, sans ordre. « Le chien mord l'homme » et « l'homme mord le chien » seraient identiques.

In [2]:
D = 512                                        # taille d'un vecteur
emb_tokens = nn.Embedding(tokenizer.get_vocab_size(), D)
emb_position = nn.Embedding(512, D)            # 512 positions maximum

ids = torch.tensor([tokenizer.encode("Le chat dort sur le canapé.").ids])
print("Ids :", ids.tolist())
print("Forme des ids :", tuple(ids.shape), "-> (1 phrase, nb de tokens)")

x = emb_tokens(ids) + emb_position(torch.arange(ids.shape[1]))
print("Forme après embeddings :", tuple(x.shape), "-> chaque token est devenu un vecteur de 512 nombres")
print("Paramètres de la table des tokens :", f"{emb_tokens.weight.numel():,}")

Ids : [[502, 14350, 257, 428, 402, 290, 2257, 381, 258, 14]]
Forme des ids : (1, 10) -> (1 phrase, nb de tokens)
Forme après embeddings : (1, 10, 512) -> chaque token est devenu un vecteur de 512 nombres
Paramètres de la table des tokens : 8,388,608


## 2. L'attention, à la main

Chaque token produit trois vecteurs :
- une **requête** (q) : « qu'est-ce que je cherche ? »
- une **clé** (k) : « qu'est-ce que je contiens ? »
- une **valeur** (v) : « ce que je transmets si on me choisit ».

Le score entre deux tokens = produit scalaire requête · clé. On passe les scores dans un **softmax** pour obtenir des poids qui somment à 1, puis chaque token reçoit la moyenne des valeurs, pondérée par ces poids.

**Le masque causal** : un token n'a pas le droit de regarder l'avenir. Sinon, pour prédire le mot suivant, il tricherait en le lisant. On met donc les scores du futur à −∞ (poids 0 après le softmax).

Regarde la matrice affichée : **le triangle du haut est à zéro**. C'est le masque.

In [3]:
T = x.shape[1]
q_proj, k_proj, v_proj = nn.Linear(D, D), nn.Linear(D, D), nn.Linear(D, D)
q, k, v = q_proj(x), k_proj(x), v_proj(x)

scores = q @ k.transpose(-2, -1) / math.sqrt(D)     # (1, T, T) : chaque token contre chaque token
masque = torch.tril(torch.ones(T, T))               # triangle du bas = autorisé
scores = scores.masked_fill(masque == 0, float("-inf"))
poids = F.softmax(scores, dim=-1)
sortie = poids @ v

mots = [tokenizer.decode([i]) for i in ids[0].tolist()]
print("Poids d'attention (ligne = token qui regarde, colonne = token regardé) :\n")
print(" " * 10 + "".join(f"{m!r:>10}" for m in mots))
for i, m in enumerate(mots):
    print(f"{m!r:>10}" + "".join(f"{p:>10.2f}" for p in poids[0, i].tolist()))
print("\nForme de la sortie :", tuple(sortie.shape), "-> même forme qu'en entrée, mais chaque vecteur a absorbé du contexte")

Poids d'attention (ligne = token qui regarde, colonne = token regardé) :

                'Le'   ' chat'      ' d'     'ort'    ' sur'     ' le'    ' can'      'ap'       'é'       '.'
      'Le'      1.00      0.00      0.00      0.00      0.00      0.00      0.00      0.00      0.00      0.00
   ' chat'      0.72      0.28      0.00      0.00      0.00      0.00      0.00      0.00      0.00      0.00
      ' d'      0.73      0.11      0.15      0.00      0.00      0.00      0.00      0.00      0.00      0.00
     'ort'      0.27      0.37      0.18      0.18      0.00      0.00      0.00      0.00      0.00      0.00
    ' sur'      0.20      0.15      0.33      0.20      0.12      0.00      0.00      0.00      0.00      0.00
     ' le'      0.05      0.17      0.19      0.23      0.21      0.16      0.00      0.00      0.00      0.00
    ' can'      0.27      0.15      0.09      0.09      0.22      0.09      0.09      0.00      0.00      0.00
      'ap'      0.03      0.13      0.

Les poids sont distribués au hasard : les couches q, k, v ne sont pas entraînées, le modèle ne sait pas encore qui regarder. Après l'entraînement, « dort » regardera fortement « chat » (qui dort ?).

Le vrai modèle utilise **plusieurs têtes d'attention en parallèle** (8 ici, de 64 dimensions chacune) : l'une peut suivre l'accord sujet-verbe, une autre les guillemets ouverts, etc.

## 3. Le modèle complet

On assemble tout dans le fichier `model.py`, enregistré sur ton Drive (la phase 4 l'importera). Un **bloc** du Transformer, c'est :

1. normalisation → **attention** → on ajoute le résultat à l'entrée (connexion résiduelle) ;
2. normalisation → **MLP** (un petit réseau qui « réfléchit » sur chaque token séparément) → on ajoute encore.

On empile **12 blocs**. À la fin, une couche transforme chaque vecteur en 16 384 scores, un par token possible : la prédiction du token suivant.

Les connexions résiduelles (`x = x + ...`) sont essentielles : elles laissent passer l'information intacte d'un bloc à l'autre, ce qui permet d'empiler beaucoup de couches sans que l'entraînement ne déraille.

`F.scaled_dot_product_attention` fait exactement le calcul de la cellule précédente (avec le masque, `is_causal=True`), mais en version optimisée pour le GPU.

In [4]:
%%writefile /content/drive/MyDrive/petit-llm/model.py
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class Config:
    vocab_size: int = 16384
    block_size: int = 512      # longueur maximale du contexte, en tokens
    n_layer: int = 12          # nombre de blocs
    n_head: int = 8            # têtes d'attention par bloc
    n_embd: int = 512          # taille des vecteurs
    dropout: float = 0.0


class Attention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.qkv = nn.Linear(c.n_embd, 3 * c.n_embd, bias=False)   # q, k, v en une seule couche
        self.proj = nn.Linear(c.n_embd, c.n_embd, bias=False)
        self.n_head, self.dropout = c.n_head, c.dropout

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # on découpe en têtes : (B, n_head, T, taille_tête)
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)            # on recolle les têtes
        return self.proj(y)


class MLP(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.fc = nn.Linear(c.n_embd, 4 * c.n_embd, bias=False)
        self.proj = nn.Linear(4 * c.n_embd, c.n_embd, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))


class Bloc(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(c.n_embd), Attention(c)
        self.ln2, self.mlp = nn.LayerNorm(c.n_embd), MLP(c)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))     # connexion résiduelle
        x = x + self.mlp(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.emb_tokens = nn.Embedding(c.vocab_size, c.n_embd)
        self.emb_position = nn.Embedding(c.block_size, c.n_embd)
        self.blocs = nn.ModuleList([Bloc(c) for _ in range(c.n_layer)])
        self.ln_final = nn.LayerNorm(c.n_embd)
        self.tete = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        # Partage de poids : la table d'entrée sert aussi de couche de sortie.
        # Même idée dans les deux sens (token <-> vecteur), et 8 M de paramètres économisés.
        self.tete.weight = self.emb_tokens.weight

        self.apply(self._init)
        for nom, p in self.named_parameters():       # initialisation réduite des sorties de bloc
            if nom.endswith("proj.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * c.n_layer))

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, ids, cibles=None):
        B, T = ids.shape
        x = self.emb_tokens(ids) + self.emb_position(torch.arange(T, device=ids.device))
        for bloc in self.blocs:
            x = bloc(x)
        logits = self.tete(self.ln_final(x))           # (B, T, vocab) : un score par token possible
        perte = None
        if cibles is not None:
            perte = F.cross_entropy(logits.view(-1, logits.size(-1)), cibles.view(-1))
        return logits, perte

    @torch.no_grad()
    def generer(self, ids, n_tokens, temperature=1.0, top_k=50):
        for _ in range(n_tokens):
            contexte = ids[:, -self.config.block_size:]
            logits, _ = self(contexte)
            logits = logits[:, -1, :] / temperature     # on ne garde que la dernière position
            if top_k:
                seuil = torch.topk(logits, top_k).values[:, [-1]]
                logits[logits < seuil] = -float("inf")  # on ne tire que parmi les k meilleurs
            suivant = torch.multinomial(F.softmax(logits, dim=-1), 1)
            ids = torch.cat([ids, suivant], dim=1)
        return ids

Writing /content/drive/MyDrive/petit-llm/model.py


## 4. Combien de paramètres, et où ?

Chaque nombre réglable du modèle est un **paramètre**. Regarde où ils se trouvent.

In [5]:
import sys
sys.path.append(DOSSIER)
from model import Config, GPT

config = Config()
modele = GPT(config)

total = sum(p.numel() for p in modele.parameters())
emb = modele.emb_tokens.weight.numel() + modele.emb_position.weight.numel()
par_bloc = sum(p.numel() for p in modele.blocs[0].parameters())
print(f"Embeddings (tokens + positions) : {emb:>12,}")
print(f"Un bloc                         : {par_bloc:>12,}")
print(f"12 blocs                        : {par_bloc * config.n_layer:>12,}")
print(f"TOTAL                           : {total:>12,}  (~{total / 1e6:.0f} M)")
print(f"\nPour comparer : ton bigramme avait 82 x 82 = {82 * 82:,} nombres.")

Embeddings (tokens + positions) :    8,650,752
Un bloc                         :    3,147,776
12 blocs                        :   37,773,312
TOTAL                           :   46,425,088  (~46 M)

Pour comparer : ton bigramme avait 82 x 82 = 6,724 nombres.


## 5. Premier passage sur de vraies données

On prend des morceaux de `train.bin`. Pour chaque morceau :
- **entrée** = les tokens 0 à 511 ;
- **cible** = les tokens 1 à 512 (les mêmes, décalés d'un cran).

Le modèle doit donc prédire, **à chaque position**, le token suivant. Un seul morceau de 512 tokens = 512 exercices de prédiction d'un coup.

La **perte** (*loss*, entropie croisée) mesure à quel point il se trompe. Un modèle qui ne sait rien répartit sa probabilité uniformément sur les 16 384 tokens : sa perte vaut alors **ln(16 384) ≈ 9,70**. C'est le point de départ attendu. L'entraînement consistera à faire descendre ce nombre.

In [6]:
donnees = np.memmap(f"{DOSSIER}/train.bin", dtype=np.uint16, mode="r")

def lot(taille=4, T=config.block_size):
    debuts = np.random.randint(0, len(donnees) - T - 1, taille)
    x = torch.stack([torch.from_numpy(donnees[i:i + T].astype(np.int64)) for i in debuts])
    y = torch.stack([torch.from_numpy(donnees[i + 1:i + 1 + T].astype(np.int64)) for i in debuts])
    return x, y

x, y = lot()
print("Entrée :", tuple(x.shape), "| Cible :", tuple(y.shape))
print("Entrée (début) :", repr(tokenizer.decode(x[0, :12].tolist())))
print("Cible  (début) :", repr(tokenizer.decode(y[0, :12].tolist())), " <- décalée d'un token")

logits, perte = modele(x, y)
print(f"\nLogits : {tuple(logits.shape)} -> pour chaque position, un score par token possible")
print(f"Perte du modèle non entraîné : {perte.item():.2f}")
print(f"Perte d'un tirage au hasard  : {math.log(config.vocab_size):.2f}")

Entrée : (4, 512) | Cible : (4, 512)
Entrée (début) : '-Pologne dans le centre-ouest de la Pologne.'
Cible  (début) : 'Pologne dans le centre-ouest de la Pologne.\n'  <- décalée d'un token

Logits : (4, 512, 16384) -> pour chaque position, un score par token possible
Perte du modèle non entraîné : 9.78
Perte d'un tirage au hasard  : 9.70


## 6. Le modèle non entraîné parle

Tout est en place : il sait prendre un début de texte et générer la suite. Mais ses paramètres sont aléatoires, donc il va produire des tokens au hasard. C'est **pire que ton bigramme** de la phase 0, qui au moins avait appris des statistiques.

À la phase 4, on verra cette sortie passer progressivement du chaos à du français.

In [7]:
debut = torch.tensor([tokenizer.encode("La ville de Paris").ids])
sortie = modele.generer(debut, 30)
print(tokenizer.decode(sortie[0].tolist()))

La ville de Paris Ré irlandais servi gagnant Luci britanniques"thaolooideoide SC servi logement auxqu doub mondial simultanément Ernst orth « Bulgarieoo murstateur commer britanniques piereres rebelles


## Bilan de la phase 3

Tu as maintenant :
- `model.py` sur ton Drive : un GPT complet de ~46 M de paramètres ;
- la preuve qu'il fonctionne de bout en bout (données → perte → génération) ;
- une perte de départ ≈ 9,70, qui servira de référence.

**Phase 4 :** la boucle d'entraînement, sur GPU cette fois.